# NiVe1303 → OSNet: подтверждающий этап

**После аудита этот этап отложен.** Следующий рекомендуемый запуск —
`../variant_16_review_protocol/train_osnet_review_protocol.ipynb`.
Этот старый протокол сохранён для воспроизводимости и требует явного `RUN_CONFIRMATION = True`.

Продолжение завершённого `nive_pilot_v1`, а не новый рецепт. Пилот не показал выигрыша:
B0 81,612%, C1 82,100%, E1 80,027% inner raw mAP@10 на одном seed.
Цель — проверить воспроизводимость этого результата на трёх seed и alternate split.

Пять завершённых стадий переиспользуются без повторных optimizer updates. Остаётся до
26 новых стадий / 37 400 updates; при скорости пилота ориентир около 9–10 часов,
но final refit может закончиться раньше. Аудит, оценка и экспорт добавляют время.

Не меняй код, budgets, seeds, устройство и RUN_NAME при продолжении. Старый notebook
с outputs остаётся историей пилота. Запускай этот notebook отдельно в прежнем .venv:
**Restart Kernel → Run All**. Автоматического продвижения в MVP и загрузки NiVe в Git нет.


In [ ]:
from pathlib import Path
import os
import sys

candidates = [Path.cwd(), Path.cwd() / 'Car-classification-MSK', *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'training/nive_transfer.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Открой notebook из проекта Car-classification-MSK')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

import torch
import pandas as pd
from IPython.display import Markdown, display
from training.audit import digest
from training.nive_transfer import VARIANT, prepare, run_experiment
from training.osnet_ablation_suite import Budget, load_json

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)
print('Repository:', REPO)
print('Torch:', torch.__version__)


## 1. Продолжаем именно завершённый пилот

Источник уже подтверждён в manifest выполненного запуска. Не ставим новое подтверждение
вручную: читаем его вместе с устройством, seed и бюджетами из этого manifest.
Если пилот отсутствует, не завершён или его подпись не совпадает, продолжение блокируется.


In [ ]:
RUN_NAME = 'nive_pilot_v1'
PHASE = 'confirm'
RUN_CONFIRMATION = False  # Долгий необязательный этап; не следующий запуск по умолчанию
ALLOW_CPU_TRAINING = False
RUN_DIR = VARIANT / 'runs' / RUN_NAME

if not (RUN_DIR / 'pilot_summary.json').is_file():
    raise RuntimeError('Сначала заверши pilot в train_nive_transfer.ipynb.')
FROZEN = load_json(RUN_DIR / 'manifest.json')
PILOT = load_json(RUN_DIR / 'pilot_summary.json')
if not PILOT.get('pilot_complete') or PILOT.get('signature') != digest(FROZEN):
    raise RuntimeError('Пилот не завершён или его manifest изменился; не начинаем другой эксперимент.')
SOURCE_CONFIRMED = FROZEN['nive']['source']['local_copy_source_confirmed_by_user']
if not SOURCE_CONFIRMED:
    raise RuntimeError('В исходном пилоте нет подтверждения источника NiVe; сначала проверь происхождение данных.')

DEVICE = FROZEN['runtime']['device']
PRETRAIN_BUDGET = Budget(**FROZEN['pretrain_budget'])
TARGET_BUDGET = Budget(**FROZEN['target_budget'])
SEEDS = tuple(FROZEN['seeds'])
NIVE_ROOT = REPO / 'NiVe1303'

print('Run:', RUN_NAME, '| phase:', PHASE, '| device:', DEVICE)
print('Source inherited from completed pilot:', FROZEN['nive']['source']['url'])
print('Seeds:', SEEDS)
print('Upper bound after the pilot: 26 new stages,',
      18 * TARGET_BUDGET.max_steps + 8 * PRETRAIN_BUDGET.max_steps, 'updates')
display(pd.DataFrame([{
    'arm': r['variant'], 'pilot mAP@10': r['inner_map'], 'best target step': r['best_step']
} for r in PILOT['primary']]))


## 2. Проверка совместимости — без обучения

Повторно проверяются данные, исходники, runtime, splits и защищённые артефакты.
При несовпадении не удаляй старый manifest и не обходи ошибку — нужен разбор причины.


In [ ]:
context = prepare(RUN_NAME, device=DEVICE, pretrain_budget=PRETRAIN_BUDGET,
                  target_budget=TARGET_BUDGET, seeds=SEEDS,
                  source_confirmed=SOURCE_CONFIRMED, nive=NIVE_ROOT)
if context['signature'] != PILOT['signature']:
    raise RuntimeError('Continuation signature differs from the completed pilot.')
print('Output:', context['output'])
print('Fingerprint:', context['signature'])
print('Preflight passed. Training has not started yet.')


## 3. Confirmation

1. Все три условия на трёх primary seed; результаты первого seed уже готовы.
2. Выбор по среднему primary raw mAP@10 и freeze медианы best step для каждого условия.
3. Все три условия на трёх alternate seed; выбор и refit budget не пересматриваются.
4. Refit на исходном outer train. Все final checkpoint фиксируются до outer evaluation.
5. Calibration-only пороги, исходная validation и три masked diagnostics, bootstrap и ONNX.

B0, C1 и E1 проверяются независимо от того, кто выиграет primary.
У C1 best target step может быть 0: это использование обученного source encoder, не пустая модель.
При прерывании — **Restart Kernel → Run All**; завершённые стадии не переобучаются.
Не запускай pilot и confirm одновременно.


In [ ]:
if not RUN_CONFIRMATION:
    raise RuntimeError('NiVe confirmation отложен после аудита. Для нового плана открой variant_16_review_protocol.')
if DEVICE == 'cpu' and not ALLOW_CPU_TRAINING:
    raise RuntimeError('CPU-обучение отключено; продолжай на устройстве исходного пилота.')
result = run_experiment(context, phase=PHASE)
print('Confirmation complete:', result['confirmation_complete'])
print('Promoted:', result['promoted'], '| Outer evaluated:', result['outer_evaluated'])


## 4. Результаты

Для анализа нужны `RESULTS.md`, `summary.json`, `selection.json`, `alternate.json`
и `paired_bootstrap.json` в том же run-каталоге.
`PILOT_RESULTS.md` и `pilot_summary.json` сохраняются отдельно.

Гипотеза о пользе внешних данных требует сравнения E1 **и с B0, и с C1**.
Прирост одного seed, отдельного masked-условия или только F1 не означает победу.
Outer validation уже использовалась раньше: это development, не независимый финальный тест.
Рецепт mixed-domain и маски частей — отдельный последующий эксперимент, не этот запуск.


In [ ]:
display(Markdown((context['output'] / 'RESULTS.md').read_text(encoding='utf-8')))
print('Detailed results:', context['output'])
